# Module 5 Worksheet — Agentic AI: ReAct, Reflexion, Multi-Agent
**Corrected in this version:** every `multimodal_chat()` call replaced with `ask()`.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../../wrapper_fix"))      # folder containing the corrected inhouse_wrappers.py
sys.path.append(os.path.abspath("../../inhouse_rag_capstone"))  # folder containing your real inhouse_llm.py

from inhouse_llm import MODEL_QWEN3_14B, MODEL_QWEN3_30B, MODEL_MISTRAL, MODEL_LLAMA, MODEL_DEVSTRAL, MODEL_QWEN2_5_VL_7B
from inhouse_wrappers import get_chat_model, InHouseEmbeddings, build_vision_messages, llm_for
from langchain_core.messages import SystemMessage, HumanMessage

embedder = InHouseEmbeddings()

def ask(system_prompt, user_prompt, model=MODEL_QWEN3_14B, max_tokens=500):
    """Correctly-routed replacement for calling multimodal_chat() directly."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]).content

def ask_vision(system_prompt, user_prompt, image_base64, model=MODEL_QWEN2_5_VL_7B, max_tokens=500):
    """Correctly-routed, correctly-formatted multimodal call."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke(build_vision_messages(system_prompt, user_prompt, image_base64)).content

print("Setup OK")

In [ ]:
from rag_pure_python import SimpleVectorStore
import json

## 1. Reproducing this module's teaser bug: agent loops without converging

In [ ]:
def calculator(expression: str) -> str:
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"Error: {e}"

VAGUE_SYSTEM = """You have a calculator(expression) tool.
Respond with ONLY one JSON object:
{"action": "calculator", "action_input": "<expr>"}
or
{"action": "final_answer", "action_input": "<answer>"}
"""
def run_buggy_agent(question, max_steps=4):
    transcript = f"Question: {question}"
    for step in range(max_steps):
        raw = ask(VAGUE_SYSTEM, transcript, max_tokens=100)
        print(f"Step {step+1}:", raw)
        try:
            action = json.loads(raw)
        except json.JSONDecodeError:
            print("parse failure, stopping")
            return None
        if action["action"] == "final_answer":
            return action["action_input"]
        observation = "ok"  # <- deliberately uninformative, reproduces the bug
        transcript += f"\nAction: {action}\nObservation: {observation}"
    return "Max steps reached (reproduced the bug)."

print(run_buggy_agent("What is 12 * 7?"))

## 2. The fix: informative observations

In [ ]:
def run_fixed_agent(question, max_steps=4):
    transcript = f"Question: {question}"
    for step in range(max_steps):
        raw = ask(VAGUE_SYSTEM, transcript, max_tokens=100)
        try:
            action = json.loads(raw)
        except json.JSONDecodeError:
            return None
        if action["action"] == "final_answer":
            return action["action_input"]
        observation = calculator(action["action_input"])  # <- actual, informative result
        transcript += f"\nAction: {action}\nObservation: {observation}"
    return "Max steps reached."

print(run_fixed_agent("What is 12 * 7?"))

## 3. Reflexion: self-critique loop
Note `model=MODEL_LLAMA` for the critique step now genuinely hits Llama's endpoint — under the old buggy code it would have silently hit Qwen3-14B.

In [ ]:
def reflexion_answer(question, max_retries=2):
    attempt = ask("Answer concisely.", question, max_tokens=150)
    for r in range(max_retries):
        critique = ask(
            "Critique this answer for correctness and completeness. If it's good, reply ONLY 'OK'. Otherwise explain the flaw.",
            f"Question: {question}\nAnswer: {attempt}",
            model=MODEL_LLAMA, max_tokens=150,
        )
        print(f"Attempt {r+1}: {attempt}\nCritique: {critique}\n")
        if critique.strip() == "OK":
            break
        attempt = ask(
            "Revise your answer based on this critique.",
            f"Question: {question}\nPrevious answer: {attempt}\nCritique: {critique}",
            max_tokens=150,
        )
    return attempt

final = reflexion_answer("What is the difference between RAG and fine-tuning?")
print("FINAL:", final)

## Teaser exercise
Build a 2-agent supervisor toy example: a 'research' agent that does knowledge_lookup, a 'math' agent that does calculator, and a supervisor that reads the question, decides which sub-agent(s) to delegate to, and merges their results into one final answer.